# 00 · EDA

Revisar los originales antes de procesarlos. Este notebook solo lee `data/raw/`.
Los conteos describen lo descargado

In [1]:
import hashlib
import json
from pathlib import Path
from urllib.parse import urlsplit

import pandas as pd
from IPython.display import display

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/raw/manifest.json").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Abre este notebook desde la carpeta del proyecto.")
RAW = RAIZ / "data" / "raw"
manifest = json.loads((RAW / "manifest.json").read_text(encoding="utf-8"))
documentos = pd.DataFrame([{k: v for k, v in d.items() if k != "archivos_raw"}
                          for d in manifest])
archivos = pd.DataFrame([dict(a, doc_id=d["doc_id"])
                        for d in manifest for a in d["archivos_raw"]])

## 1. Inventario

In [2]:
archivos["formato"] = archivos["archivo"].map(lambda p: Path(p).suffix.lower())
display(pd.Series({
    "Documentos": len(documentos),
    "Archivos": len(archivos),
    "Tamaño declarado (MiB)": round(archivos["bytes"].sum() / 1024**2, 2),
}, name="Inventario").to_frame())
display(archivos["formato"].value_counts().rename("archivos").to_frame())
display(documentos[["doc_id", "titulo", "fuente", "url"]].head(5))

,Inventario
Documentos,282.00
Archivos,282.00
Tamaño declarado (MiB),116.85


,archivos
formato,
.html,241
.pdf,41


,doc_id,titulo,fuente,url
0,auto_cc_a841_2025,Auto 841 de 2025 - Ley 2381 de 2024,Corte Constitucional - Relatoría,https://www.corteconstitucional.gov.co/relator...
1,co_constitucion_1_1991,Constitución Política de Colombia,Departamento Administrativo de la Función Públ...,https://www.funcionpublica.gov.co/eva/gestorno...
2,co_decreto_1069_2015,Decreto Único Reglamentario del Sector Justici...,Departamento Administrativo de la Función Públ...,https://www.funcionpublica.gov.co/eva/gestorno...
3,co_decreto_1082_2015,Decreto Único Reglamentario del Sector Adminis...,Departamento Administrativo de la Función Públ...,https://www.funcionpublica.gov.co/eva/gestorno...
4,co_decreto_1083_2015,Decreto Único Reglamentario del Sector de Func...,Departamento Administrativo de la Función Públ...,https://www.funcionpublica.gov.co/eva/gestorno...


## 2. Cobertura

Un documento puede pertenecer a varias áreas. La suma por área puede superar el total.

In [3]:
display(documentos["areas"].explode().value_counts().rename("documentos").to_frame())
display(documentos["tipo"].value_counts().rename("documentos").to_frame())
display(documentos["anio"].value_counts().sort_index().rename("documentos").to_frame())
dominios = documentos["url"].map(lambda url: urlsplit(url).netloc.lower())
display(dominios.value_counts().rename("documentos por dominio").to_frame())

,documentos
areas,
constitucional,74
procesal,55
administrativo,44
laboral,43
penal,40
civil,38
familia,30
mercados,25
tributario,20


,documentos
tipo,
ley,171
sentencia,84
decreto,23
decision,2
auto,1
constitucion,1


,documentos
anio,
1873,1
1887,2
1890,1
1932,1
1936,1
1948,1
1950,1
1959,1
1968,1


,documentos por dominio
url,
www.funcionpublica.gov.co,198
www.corteconstitucional.gov.co,47
cortesuprema.gov.co,18
www.cancilleria.gov.co,2
normas.cra.gov.co,2
normograma.dian.gov.co,2
www.consejodeestado.gov.co,2
www.mintic.gov.co,1
normograma.superservicios.gov.co,1


## 3. Metadatos y procedencia

La fecha de consulta no demuestra vigencia. Los originales marcados `redistribuir_raw=False` requieren revisar sus anotaciones antes de publicarlos.

In [4]:
campos = ["doc_id", "titulo", "tipo", "numero", "anio", "organo_emisor",
          "areas", "fuente", "url", "fecha_consulta", "vigencia", "licencia_fuente"]
faltantes = pd.Series({campo: sum(d.get(campo) in (None, "", []) for d in manifest)
                      for campo in campos}, name="faltantes")
display(faltantes.to_frame())
display(documentos.loc[documentos["doc_id"].duplicated(keep=False), ["doc_id", "titulo"]])
display(archivos.loc[archivos["sha256"].duplicated(keep=False), ["doc_id", "archivo", "sha256"]])
display(documentos["vigencia"].value_counts().rename("documentos").to_frame())
revisar_publicacion = documentos.loc[documentos["redistribuir_raw"] != True,
                                    ["doc_id", "titulo", "redistribuir_raw", "edicion_con_anotaciones"]]
display(revisar_publicacion)

,faltantes
doc_id,0
titulo,0
tipo,0
numero,0
anio,0
organo_emisor,0
areas,0
fuente,0
url,0
fecha_consulta,0


,doc_id,titulo


,doc_id,archivo,sha256


,documentos
vigencia,
por_verificar,275
alcance_y_efectos_de_la_providencia,5
requiere_control_temporal_A841_2025_C264_2026,1
consultar_decision_C481_2019_y_efectos_temporales,1


,doc_id,titulo,redistribuir_raw,edicion_con_anotaciones
35,decision_351_1993,Régimen común sobre derecho de autor y derecho...,False,True
36,decision_486_2000,Régimen común de propiedad industrial,False,True
62,ley_1835_2017,Remuneración a autores de obras cinematográficas,False,True
63,ley_1878_2018,Modificaciones al Código de Infancia y Adolesc...,False,True
67,ley_2089_2021,"Prohibición del castigo físico a niñas, niños ...",False,True
89,ley_675_2001,Régimen de propiedad horizontal,False,True
94,ley_84_1873,Código Civil colombiano,False,True
218,ley_1943_2018,Ley de financiamiento de 2018; fuente histórica,False,True
241,ley_28_1932,Régimen patrimonial en el matrimonio,False,True
265,sentencia_ce_suj4005_2020,2020CE-SUJ-4-005 - Deducciones tributarias,False,True


## 4. Integridad

Comparar cada archivo con su tamaño y SHA-256 registrados. Esto verifica la copia, no la corrección jurídica.

In [5]:
def revisar_archivo(archivo):
    ruta = (RAW / archivo["archivo"]).resolve()
    if not ruta.is_relative_to(RAW.resolve()):
        return "ruta fuera de raw"
    if not ruta.is_file():
        return "no encontrado"
    contenido = ruta.read_bytes()
    if not contenido:
        return "vacío"
    if len(contenido) != archivo["bytes"]:
        return "tamaño distinto"
    if hashlib.sha256(contenido).hexdigest() != archivo["sha256"]:
        return "hash distinto"
    return "ok"

integridad = archivos[["doc_id", "archivo"]].copy()
integridad["estado"] = [revisar_archivo(a) for a in archivos.to_dict("records")]
display(integridad["estado"].value_counts().rename("archivos").to_frame())
display(integridad.loc[integridad["estado"] != "ok"])

,archivos
estado,
ok,282


,doc_id,archivo,estado


## Resultados y decisiones

